# 🗺️ 00 — Orientation: the story, the map, and how this course works

Welcome. This is the first of fifteen notebooks that teach you an entire working
system — AI agents that buy network services from each other and pay with tokens — by
**poking at the real code**, never at simplified copies.

**What you'll be able to do after this notebook**

- retell the project's story — Ada, Bell, ticket #7 — in your own words, with the numbers right
- run, edit, restart, and (deliberately) break a Jupyter notebook without fear
- explain what `import a2a_interfaces` actually does, and point at the exact file it loads
- draw the repo's package map as **trust domains** and say which way imports flow — and why
- check your own machine and know exactly which notebooks it can run today

**You need:** nothing but this repo with its Python workspace synced
(`uv sync --all-packages` — section 3 explains what that sentence means). No blockchain,
no router lab, no LLM endpoint. Every notebook in this course runs without them.

**Estimated time:** 45–60 minutes.

> **How to run:** in VS Code or Jupyter, pick the repo's `.venv` kernel, then press
> **Shift+Enter** on each cell, top to bottom. If that instruction meant nothing to you,
> perfect — section 1 exists for you.
>
> Companions for later (don't read them yet):
> [docs/00-the-story.md](../../../docs/00-the-story.md) — the full story ·
> [docs/02-architecture.md](../../../docs/02-architecture.md) — the map, formally ·
> [docs/LEARNING-PATH.md](../../../docs/LEARNING-PATH.md) — the reading route this
> course is the executable twin of.

## 1. What is this document you're reading?

A **Jupyter notebook** is a document made of **cells**. A cell holds either prose (like
this one — a *markdown* cell; double-click it to see its raw text) or runnable Python
(a *code* cell). Behind the document sits a **kernel**: one live Python process that
executes whatever code you send it and remembers everything between cells.

Three verbs cover most of notebook life:

- **Run** a cell (`Shift+Enter`): the kernel executes it; anything it *prints* — plus
  the value of the cell's *last expression* — appears below it.
- **Run All**: run every cell top to bottom. Every notebook in this course survives that.
- **Restart** the kernel: kill the Python process, start a fresh one that remembers
  *nothing*. The cure for a confused notebook is always: restart, then Run All.

Run your first cell and check that both kinds of output show up.

In [ ]:
print("this line was PRINTED by the cell")           # output kind 1: an explicit print

"this string is the cell's LAST EXPRESSION"          # output kind 2: displayed automatically

The kernel **remembers**. The next cell defines two variables and nothing more; section
2 will use them — proof that all cells share one living Python process. Order matters:
after a kernel restart, running section 2 *before* this cell fails with a `NameError`.
That's not a bug; that's the deal you make with a notebook.

In [ ]:
dataset_gb = 45        # Ada's job, arriving in section 2 — the kernel will remember this
deadline_hours = 2
print("stored:", dataset_gb, "GB to move within", deadline_hours, "hours")

One more survival skill: reading an **error**. When code fails, Python prints a
**traceback** — a report naming the error type, the message, and the line that raised
it. It looks scary; it is just the answer to "what went wrong, where". Below we trigger
one *on purpose* and catch it with `try/except` — a construct meaning "attempt this; if
it raises an error, run that instead of dying".
[01_python_toolbox.ipynb](01_python_toolbox.ipynb) teaches it properly; this course uses
it whenever we break something deliberately, so the notebook keeps running.

In [ ]:
try:
    print(a_name_nobody_defined)
except NameError as err:
    print("the kernel raised:", type(err).__name__, "→", err)

print("…and the notebook is still alive. Deliberate breakage is how we learn here.")

**✏️ Your turn 1.1 — Predict the error type**

Section 1's breakage was a `NameError`. Now make Python refuse something *different*: adding a number to a piece of text. Write the error type's name into `my_prediction` BEFORE running, then run and compare. Success: the un-commented self-check passes — your prediction matches what the kernel actually raised.

In [ ]:
my_prediction = "SomeError"   # TODO: replace with the error name you expect (spelling counts)

try:
    dataset_gb + " GB"        # a number plus text — Python refuses to guess what you meant
except Exception as err:
    caught = type(err).__name__
    print("the kernel raised:", caught, "→", err)

# TODO: un-comment the self-check once your prediction is in:
# assert my_prediction == caught, f"you predicted {my_prediction}, the kernel said {caught}"

<details><summary>✅ Solution 1.1 — peek only after trying</summary>

```python
my_prediction = "TypeError"
```

Mixing types is a `TypeError` — a *type* problem (int + text), where section 1's `NameError` was a *name* problem (the variable didn't exist); the traceback's first word always tells you which family you're in.

</details>

## 2. The story, in one screen

At 13:30, **Ada** — an AI agent: a program with an LLM for a brain and a crypto wallet
for a hand — is handed a job: move a **45 GB** dataset from host-A to host-B, **done
before 16:00**. The public network *might* be fast enough; "might" is not a plan. She
needs a guarantee. **Bell** — another AI agent, working for a network operator — sells
exactly that: guaranteed bandwidth, by the hour.

Today that sale takes humans: calls, contracts, an API key by email on Thursday. Ada has
two and a half hours. We want her to buy the guarantee **by herself, in seconds, from a
stranger** — and then we want the network to actually obey. That sentence hides two hard
problems, and this repo is a careful answer to both:

1. **The fair-exchange problem.** How do two strangers swap money for a promise without
   one robbing the other? Answer: a *vending machine* — a smart contract where
   payment-and-ticket is one atomic, all-or-nothing motion.
2. **The obedient-network problem.** A purchase is just data; how does it become
   *physics* — actually-configured routers? Answer: a deterministic *bouncer* — the
   controller — that checks the ticket and directs the router-driving *hands*.

First, check Ada's arithmetic yourself. The kernel still remembers `dataset_gb` from
section 1. (`assert` checks a claim and stops loudly if it's false — this course pins
every factual claim with one, so a broken claim breaks the notebook.)

In [ ]:
assert dataset_gb == 45, "section 1's cells must run first — Run All from the top"

bits_to_move = dataset_gb * 1_000_000_000 * 8   # GB → bytes → bits (8 bits per byte)
seconds = deadline_hours * 3600
bits_per_second = bits_to_move // seconds       # // = whole-number division

print("bits to move   :", bits_to_move)
print("seconds allowed:", seconds)
print("needed rate    :", bits_per_second, "bits/s =", bits_per_second // 1_000_000, "Mbps  ← exactly")
assert bits_per_second == 50_000_000

**50 Mbps, exactly** — the numbers were chosen so the story's arithmetic is honest.
Here is the whole play, condensed from the epilogue of
[docs/00-the-story.md](../../../docs/00-the-story.md). Every value below is canonical;
section 4 shows you their single source of truth:

```
13:31  Bell signs an offer: 50 Mbps · 14:00–16:00 · 10 TOK       (a signed catalog page)
13:32  Ada's LLM decides: {"accept": true, …}                    (judgment slot 1 of 2)
13:32  fulfill() — ONE transaction: 10 TOK → Bell,
       ticket #7 → Ada. All or nothing.                          (the vending machine's clunk)
14:02  Ada proves to the bouncer she owns #7                     (fresh challenge; no key ever travels)
14:02  Five-line checklist passes → translator → the router
       now rate-limits A→B at 50 Mbps                            (the ticket became physics)
16:00  Chain time passes the window's end → torn down            (expired coupon)
  and  Bell can REVOKE mid-window — the kill-switch that
       proves the ticket, not goodwill, governs the wire
```

Ticket **#7** is an *entitlement*: a one-of-a-kind, ownable bundle of terms — an NFT
used as a land-registry row, not as monkey art. And this is the sentence that carries
the entire course. If you keep one thing, keep this:

> **A token is the right to push one config to the router, and only deterministic
> code — never an LLM — may honor it.**

Every notebook that follows makes one clause of that sentence concrete. And the repo's
punchline: a *second*, totally different service (telemetry, sold by Tess) flows through
the **same** machinery with only the last-mile translator swapped. That is the thesis —
the settlement layer is service-agnostic.

**✏️ Your turn 2.1 — Squeeze Ada's deadline**

The 50 Mbps came from 45 GB ÷ 2 hours. Change `tight_hours` to `1` and rerun. Success: the un-commented self-check passes at exactly 100 Mbps — the arithmetic, not the storyteller, is what moved.

In [ ]:
tight_hours = 2      # TODO: change to 1 — half the time, then rerun
tight_rate = (dataset_gb * 1_000_000_000 * 8) // (tight_hours * 3600)
print(f"{dataset_gb} GB in {tight_hours} h → needs {tight_rate // 1_000_000} Mbps")

# TODO: un-comment once tight_hours is 1:
# assert tight_rate == 100_000_000, "45 GB in 1 hour is exactly double the story's rate"

<details><summary>✅ Solution 2.1 — peek only after trying</summary>

```python
tight_hours = 1
# → 45 GB in 1 h → needs 100 Mbps
```

Halving the window doubles the required bits-per-second — which is why Bell sells a *rate* and a *time window* together: the two trade off exactly.

</details>

## 3. Terminal commands, uv, and the workspace

Some things happen outside the notebook, in a **terminal** — a window where you type a
command, press Enter, and a program runs and prints text back. In this course, fenced
blocks like the one below are terminal commands, *not* notebook cells:

```bash
uv sync --all-packages     # run once, in a terminal, at the repo root
```

Four glosses, two sentences each:

- A **terminal command** is a program's name followed by its arguments, executed by a
  shell. `uv sync --all-packages` runs the program `uv` and tells it to install everything.
- **uv** is the Python toolchain this repo uses for *everything*: installing
  dependencies, pinning exact versions in `uv.lock`, and running code. Any repo Python
  you run from a terminal gets prefixed with `uv run`.
- A **virtual environment** (the `.venv/` folder at the repo root) is a private,
  per-project Python installation, so this repo's libraries can never collide with
  another project's. The kernel executing these cells lives inside it.
- A **workspace** is uv's way of managing several related packages in one repo against
  one shared lockfile. This repo's workspace has six Python packages resolving together
  (section 5 counts them, live).

Don't take the venv claim on faith — ask the kernel which Python it actually is.

In [ ]:
import sys
from pathlib import Path

print("this kernel's Python:", sys.executable)
print("Python version      :", sys.version.split()[0])
assert ".venv" in sys.executable, "not the workspace venv — pick the .venv kernel (or run via uv run)"
print("✓ the kernel lives in the repo's .venv — the virtual environment is real")

**✏️ Your turn 3.1 — Walk up to the venv folder**

`sys.executable` is a *file* deep inside the venv. Using `.parent` (a `Path`'s "one folder up"), make `venv_dir` point at the folder literally named `.venv`. Success: the un-commented self-check passes.

In [ ]:
exe = Path(sys.executable)     # …/.venv/bin/python — a file, some levels deep
venv_dir = exe                 # TODO: climb with .parent until the folder is named ".venv"
print("start:", exe)
print("yours:", venv_dir, "→ named:", venv_dir.name)

# TODO: un-comment when venv_dir is the venv itself:
# assert venv_dir.name == ".venv", "keep climbing — each .parent goes one folder up"

<details><summary>✅ Solution 3.1 — peek only after trying</summary>

```python
venv_dir = exe.parent.parent   # python → bin → .venv
```

A path is data you can compute with — two `.parent` hops turn the interpreter's file into the environment's folder, no text surgery needed.

</details>

## 4. Modules, packages, imports — from zero

Three glosses:

- A **module** is a single `.py` file of Python code.
- A **package** is a folder of modules (marked by an `__init__.py` file), importable as
  a unit.
- **`import`** loads a module — runs its file, once — and binds it to a name in your
  kernel, so you reach everything it defines with a dot: `module.thing`.

Start with a module Python ships with: `json`. (**JSON** is a plain-text format for
structured data — you'll live in it from notebook 02 on.) The point to notice: a module
is *literally a file*, and Python will tell you which one.

In [ ]:
import json

print("imported  :", json.__name__)
print("which file:", json.__file__, " ← a module IS a file on disk")
print("using it  :", json.dumps({"ticket": 7, "price_tok": 10}))

Now the repo's own foundation package — and a naming subtlety worth meeting early. The
repo *folder* is `interfaces/`, but you type `import a2a_interfaces`. The import name
comes from the package's code layout, not from the folder that happens to contain it.
The next cell imports it, asks the same question — *which file did that load?* — and
locates the repo root so we can compare paths exactly.

In [ ]:
import a2a_interfaces

# find the repo root: walk upward from this notebook's folder until we hit .git
ROOT = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents] if (p / ".git").exists())

loaded = Path(a2a_interfaces.__file__)
print("repo root  :", ROOT)
print("file loaded:", loaded)
assert loaded.resolve() == (ROOT / "interfaces" / "src" / "a2a_interfaces" / "__init__.py").resolve()
print("✓ `import a2a_interfaces` executed interfaces/src/a2a_interfaces/__init__.py")

How did the import find `interfaces/src/a2a_interfaces/`? Two pieces:

- **src-layout.** Each package keeps its importable code under
  `<folder>/src/<import_name>/`. The folder name (`interfaces`) is for humans browsing
  the repo; the import name (`a2a_interfaces`) is declared in
  `interfaces/pyproject.toml` and is what Python sees. Parking code under `src/` also
  guarantees you always import the *installed* package — never a same-named file that
  happens to sit in your current directory.
- **Editable installs.** `uv sync` installed every workspace member in *editable* mode:
  instead of copying code into `.venv`, it planted a pointer back to the source folder.
  That is why `__file__` points at the real repo source — and why editing that source
  takes effect on your next kernel restart, with no re-install.

So what did the import actually hand us? `dir(x)` lists every name an object carries —
for a package, that's what it exports.

In [ ]:
names = [n for n in dir(a2a_interfaces) if not n.startswith("_")]   # _names are internal
print("a2a_interfaces exports", len(names), "public names — the first few:")
for n in names[:8]:
    print("   ", n)
print("    …")
assert "Offer" in names and "EntitlementReader" in names 
print("✓ 'Offer' (a shape → notebook 02) and 'EntitlementReader' (a port → notebook 03) are both in there")

**✏️ Your turn 4.1 — Predict the second service's shape**

Section 2's punchline promised a *second* service — telemetry, sold by Tess — flowing through the same machinery. Predict whether the treaty already exports a `TelemetryNeed` shape: write `"yes"` or `"no"` into `my_guess`, then run. Success: the un-commented self-check passes.

In [ ]:
my_guess = "?"    # TODO: "yes" or "no" — does the treaty already carry TelemetryNeed?

exported = "TelemetryNeed" in names
print("TelemetryNeed exported:", exported)

# TODO: un-comment once your guess is in:
# assert my_guess == ("yes" if exported else "no"), "the treaty disagrees with you"

<details><summary>✅ Solution 4.1 — peek only after trying</summary>

```python
my_guess = "yes"
```

The telemetry shapes live in bedrock from day one — that is what makes the "same machinery, different last mile" thesis checkable rather than aspirational.

</details>

One more import form you'll see everywhere: `from X import Y as Z` reaches *into* a
package and binds one piece of it under a name you choose. The piece we grab —
`a2a_interfaces.fixtures` — is the **canonical example**: the one module where Ada's and
Bell's exact values live (addresses, ticket number, price, window). Story, docs, and
tests all quote from it, so they can never disagree.
[04_the_canonical_example.ipynb](04_the_canonical_example.ipynb) dissects every value;
today we only shake hands — and pin section 2's arithmetic against the real constant.

In [ ]:
from a2a_interfaces import fixtures as fx

print("Ada  (buyer)   :", fx.ADA, " ← an *address*: an account id on the chain (notebook 06)")
print("Bell (provider):", fx.BELL)
print("ticket id      :", fx.TICKET_ID)
print("capacity       :", fx.CAPACITY_50_MBPS, "bits/s")
print("window length  :", fx.WINDOW.end - fx.WINDOW.start, "seconds")

assert fx.TICKET_ID == 7
assert fx.CAPACITY_50_MBPS == bits_per_second, "story arithmetic and repo constant must agree"
assert fx.WINDOW.end - fx.WINDOW.start == seconds
print("✓ section 2's hand-computed numbers equal the repo's canonical constants")

Break it on purpose: import a package that doesn't exist. `ModuleNotFoundError` is the
face you'll meet whenever a name is misspelled — or whenever the workspace isn't synced.
Recognizing it will save you an hour someday.

In [ ]:
try:
    import a2a_interface   # typo: missing the final 's'
except ModuleNotFoundError as err:
    print("✗ import failed, as expected →", err)
finally:
    print("the fix is always one of: fix the spelling, or run `uv sync --all-packages`")

**✏️ Your turn 4.2 — Misspell the attribute, not the module**

You just met `ModuleNotFoundError` (a bad *module* name). Now break the other half of `module.thing`: the guess `fx.PRICE` below is wrong — the real constant is `PRICE_10_TOK`. Run the cell, read the traceback's first word, and copy the error type's name into `error_name`. Success: the self-check passes — and you know which of the two errors points at which half of a dotted name.

In [ ]:
error_name = "?"                     # TODO: run once, read the error, write its name here

try:
    fx.PRICE                         # wrong guess — the real name is fx.PRICE_10_TOK
except Exception as err:
    caught = type(err).__name__
    print("the kernel raised:", caught, "→", err)

print("the real constant:", fx.PRICE_10_TOK, "← integer money; notebook 04 explains why it's huge")

# TODO: un-comment once error_name is filled in:
# assert error_name == caught, f"the traceback said {caught}"

<details><summary>✅ Solution 4.2 — peek only after trying</summary>

```python
error_name = "AttributeError"
```

A dotted name fails in two ways: the module half missing is `ModuleNotFoundError`, the attribute half missing is `AttributeError` — the error type tells you which half to fix.

</details>

## 5. The repo map: trust domains, not size buckets

Time to zoom out. The repo splits into packages, and here is the idea most people miss:
the boundaries are **trust domains**. Each package exists because it is trusted with
exactly one dangerous thing — something every other package must *never* touch:

| Package | Story role | Trusted with… |
|---|---|---|
| `interfaces` | the treaty | the shared shapes + ports every border agrees on — and nothing else |
| `contracts` | the vending machine | money movement and ticket existence (Solidity, on-chain) |
| `chainmcp` | the banking app | **the private keys** — the only package that ever signs |
| `netlab` | the miniature internet | the router topology (containerlab YAML, not Python) |
| `netctl` | the hands | speaking gNMI to routers — knows no tickets, no chains |
| `controller` | the bouncer | the authorization decision — deterministic, never an LLM |
| `agents` | the brains | LLM judgment, at exactly two decision points |
| `e2e` | the stage | wiring it all together: skeleton, tests, dashboard |

(A ninth folder, `llmserve`, holds deployment scripts for the agents' LLM —
infrastructure that is never imported, so it lives outside the import map.)

Trust the disk over any table — including that one. Check the folders exist:

In [ ]:
roles = {
    "interfaces": "the treaty",        "contracts": "the vending machine",
    "chainmcp":   "the banking app",   "netlab":    "the miniature internet",
    "netctl":     "the hands",         "controller": "the bouncer",
    "agents":     "the brains",        "e2e":       "the stage",
    "llmserve":   "LLM deploy infra (not imported)",
}
for name, role in roles.items():
    assert (ROOT / name).is_dir(), name + " missing?!"
    print("✓", name.ljust(10), "→", role)
print("\nall nine folders exist on disk — the map is not aspirational")

Which of those are *Python*? The workspace's member list answers — and notice it is
**configuration, not convention**: the architecture diagram and the build config are
the same statement. `pyproject.toml` is written in TOML (a config file format Python
reads with the standard `tomllib` module).

In [ ]:
import tomllib

members = tomllib.loads((ROOT / "pyproject.toml").read_text())["tool"]["uv"]["workspace"]["members"]
print("Python workspace members:", members)
assert len(members) == 6
print("✓ 6 of 9 domains are Python; contracts (Solidity), netlab (topology YAML),")
print("  and llmserve (deploy scripts) are trust domains without Python code")

**✏️ Your turn 5.1 — Compute the non-Python domains**

The table claimed 9 trust domains; the workspace answered 6 Python members. Compute the difference yourself: make `non_python` the set of folders in `roles` that are NOT workspace members (hint: `set(roles) - set(members)`). Success: exactly the three domains the previous cell named.

In [ ]:
non_python = set()        # TODO: the trust domains that are not Python packages
print("trust domains without Python:", sorted(non_python))

# TODO: un-comment once computed:
# assert non_python == {"contracts", "netlab", "llmserve"}

<details><summary>✅ Solution 5.1 — peek only after trying</summary>

```python
non_python = set(roles) - set(members)
```

Set difference over the real config beats re-typing the table — the disk and `pyproject.toml` stay the single source of truth.

</details>

One rule makes the map safe to build on: **imports flow downward only**. A package may
import packages below it — never above it, never in a cycle. `interfaces` is bedrock: it
depends on (almost) nothing, so everything can stand on it. `e2e` is the peak: it may
import everything, and nothing imports it. Picture water flowing downhill:

In [ ]:
print(r'''
                     e2e                  ← the stage: may import everything
        ┌─────────┬───┴────┬─────────┐
     agents  controller  chainmcp  netctl        netlab: no Python, no arrows
        │         │        │  └╌╌→ contracts     (ABI only: the compiled artifact, not code)
        │         │        │         │
        └─────────┴───┬────┴─────────┘
                      ▼
                 interfaces               ← bedrock: shapes + ports; imports NOTHING upward
''')
print("every arrow points down — that direction IS the architecture")

Claims deserve proof — the house religion here is *evidence or it didn't happen*, and
you'll keep hearing it. So prove a boundary live: importing bedrock must not drag in
machinery from the layers above. We imported `a2a_interfaces` back in section 4; if it
had secretly pulled in `web3` (blockchain I/O), `pygnmi` (router I/O), or `fastapi`
(HTTP serving), those names would now sit in `sys.modules` — the kernel's registry of
everything loaded so far. (The `f"…"` strings below are *f-strings* — they paste values
into text; notebook 01 opens with them.)

In [ ]:
for heavy in ["web3", "pygnmi", "fastapi", "langgraph"]:
    loaded = heavy in sys.modules
    print(f"{heavy:<10} {'✗ LOADED — boundary broken!' if loaded else '✓ not loaded'}")
    assert not loaded

print("\n✓ bedrock pulled in no machinery from above — the treaty is pure paper")

## 6. The ten hard rules

The trust domains are guarded by ten written rules in
[`CLAUDE.md`](../../../CLAUDE.md) — the repo's contract with anyone, human or AI, who
edits it. Breaking one is defined as a wrong answer *even if the code works*, because
each rule protects a boundary the system's trustworthiness depends on. Don't take my
summary — read their opening words straight out of the file, live:

In [ ]:
import re

claude_md = (ROOT / "CLAUDE.md").read_text()
hard_rules = claude_md.split("## Hard rules")[1].split("\n## ")[0]   # just that section
leads = re.findall(r"^\d+\.\s+\*\*(.+?)\*\*", hard_rules, flags=re.M)

for i, lead in enumerate(leads, 1):
    print(f"{i:>2}. {lead}")
assert len(leads) == 10
print("\n✓ ten rules, read from CLAUDE.md itself — not paraphrased from memory")

In beginner words — one sentence and one *why* each:

| # | In plain words | Why it exists |
|---|---|---|
| 1 | Authorization is plain code; LLMs judge only "should I buy?" (Ada) and "should I quote?" (Bell). | A creative bouncer can be sweet-talked; arithmetic can't. |
| 2 | Private keys exist inside `chainmcp` and nowhere else — everyone else verifies, never signs. | A key that travels is a key that leaks, and the key *is* the identity. |
| 3 | Cross-package data shapes come only from `a2a_interfaces`, and changing one bumps its version. | Two definitions of one payload drift silently — until they disagree in production. |
| 4 | The controller's decision code imports no I/O whatsoever. | Pure logic tests in microseconds and can't be corrupted by the outside world. |
| 5 | Chain time is the only clock that decides validity. | Three machines have three clocks; seconds of drift make unreproducible bugs. |
| 6 | `netctl` receives device names, never ticket names — the secret map lives in one controller file. | The hands must work for any topology (ADR-005). |
| 7 | Mocks implement the same Protocol as real adapters and pass the same shared tests. | A fake that behaves differently makes every green test a lie. |
| 8 | Teardown is idempotent: running it twice is a success, not an error. | Cleanup runs during failures; choking on "already clean" wedges the system. |
| 9 | No new abstraction with a single implementation. | Speculative layers cost understanding today for flexibility that never comes. |
| 10 | Evidence or it didn't happen: every milestone ends with real, pasted output. | "It works" is a claim; a green run you can read is a fact. |

Don't memorize them. You'll *watch each one be enforced in code* as the course runs —
rule 4 is proven with an AST scan in
[08_controller_the_bouncer.ipynb](08_controller_the_bouncer.ipynb), rule 7 live in
[09_netctl_the_hands.ipynb](09_netctl_the_hands.ipynb), rule 8 by calling teardown
twice in [05_the_walking_skeleton.ipynb](05_the_walking_skeleton.ipynb)… But one of
them we can verify **right now**.

Rule 2 says only `chainmcp` holds keys — which implies only `chainmcp` should even
*speak* to the chain. Scan every Python package's real source tree for imports of
`web3`, the library that talks to a blockchain (notebook 06 introduces it properly):

In [ ]:
importers = set()
for pkg in members:
    for f in (ROOT / pkg / "src").rglob("*.py"):
        text = f.read_text()
        if "import web3" in text or "from web3" in text:
            importers.add(pkg)

print("packages whose source imports web3:", importers)
assert importers == {"chainmcp"}
print("✓ verified against the actual source tree: only the banking app talks to the chain")

**✏️ Your turn 6.1 — Point the scanner at the brains**

Rule 1 confines LLM judgment to `agents`. The scanner below is still aimed at `web3`; change `needle` to `"langgraph"` (the library the agents' LLM graphs are built on) and rerun. Success: the un-commented self-check passes — exactly one package thinks in graphs.

In [ ]:
needle = "web3"                       # TODO: change to "langgraph", then rerun
found = set()
for pkg in members:
    for f in (ROOT / pkg / "src").rglob("*.py"):
        text = f.read_text()
        if f"import {needle}" in text or f"from {needle}" in text:
            found.add(pkg)
print(f"packages importing {needle}:", found)

# TODO: un-comment once needle is "langgraph":
# assert found == {"agents"}, "only the brains may import the graph library"

<details><summary>✅ Solution 6.1 — peek only after trying</summary>

```python
needle = "langgraph"   # → {'agents'}
```

The same three-line scan verifies any "only X imports Y" boundary — you just produced evidence for rule 1 exactly the way the cell above did for rule 2.

</details>

## 7. How this course works: four surfaces, one order

This repo believes you learn by **doing**: watch it happen → poke it yourself → touch
the real thing. Four hands-on surfaces support that rhythm, and this course is the
spine:

1. **This course** (`e2e/notebooks/course/00…14`) — progressive, from absolute zero,
   every term glossed. Do these in order; each assumes everything before it and nothing
   after it.
2. **The explore notebooks** (`e2e/notebooks/*_explore.ipynb`) — compact per-component
   tours at working-engineer altitude. Each course chapter links its twin as the
   "deeper dive".
3. **The scratch bench** (`e2e/notebooks/scratch_inspect.ipynb`) — pre-wired imports,
   deliberately empty. Your own questions go there.
4. **The cast labs** (`contracts/EXPLORE*.md`) — the Solidity surface: terminal recipes
   you type against a live local chain (`forge inspect`, `cast call`). Notebook 06
   sends you there.

All four exist on disk right now — check:

In [ ]:
explore = sorted(p.name for p in (ROOT / "e2e" / "notebooks").glob("*_explore.ipynb"))
labs = sorted(p.name for p in (ROOT / "contracts").glob("EXPLORE*.md"))
scratch = (ROOT / "e2e" / "notebooks" / "scratch_inspect.ipynb").exists()

print("explore notebooks:", len(explore))
for n in explore:
    print("   ", n)
print("cast labs        :", len(labs), "→", ", ".join(labs))
print("scratch bench    :", "✓ present" if scratch else "✗ missing")
assert len(explore) >= 7 and len(labs) >= 4 and scratch

**✏️ Your turn 7.1 — Find a chapter's deeper-dive twin**

Every course chapter names an explore-notebook twin as its deeper dive. Filter the `explore` list down to the bouncer's: keep only the names containing `"controller"` (hint: `[n for n in explore if "controller" in n]`). Success: a one-item list.

In [ ]:
twin = explore     # TODO: keep only the names containing "controller"
print("your filter returns:", twin)

# TODO: un-comment once filtered:
# assert twin == ["controller_explore.ipynb"], "expected exactly the bouncer's twin"

<details><summary>✅ Solution 7.1 — peek only after trying</summary>

```python
twin = [n for n in explore if "controller" in n]
```

A list comprehension is a filter you can read aloud — and chapter 08 will send you to exactly this file as its deeper dive.

</details>

### The full path (15 notebooks)

| # | Notebook | You come out able to… | Extra infra |
|---|---|---|---|
| 00 | 🗺️ orientation | retell the story, navigate the repo, run cells, explain module/import/trust domain | — |
| 01 | 🧰 python_toolbox | read every language construct the repo uses: type hints, decorators, dataclasses, Enums, exceptions, `with`, threads | — |
| 02 | 🧬 pydantic_from_zero | explain how every cross-package shape validates itself at the border | — |
| 03 | 🔌 protocols_and_ports | satisfy a `Protocol` without inheriting — and say why the architecture swaps on that trick | — |
| 04 | 🎭 the_canonical_example | dissect every canonical value: addresses, unix windows, integer money, the ABI blob by hand | — |
| 05 | 🎬 the_walking_skeleton | run the entire lifecycle on cardboard fakes; name every check in the naive predicate | — |
| 06 | ⛓️ blockchain_from_zero | derive an address from a key, drive a disposable chain, read `Settlement.sol` section by section | Anvil |
| 07 | 🖋️ chainmcp_the_signing_adapter | sign an EIP-712 offer in Python the Solidity contract accepts — digests matched byte-for-byte | Anvil |
| 08 | 🕴️ controller_the_bouncer | walk the real predicate's every deny path, defeat a replay attack, drive the HTTP API in-process | — |
| 09 | 🛠️ netctl_the_hands | explain gNMI/YANG paths, drive the mock provisioner, read the real one honestly (ADR-006) | lab optional |
| 10 | 🤖 agents_the_brains | run the LangGraph agents on a stub LLM; point at the exactly-two judgment cells | — |
| 11 | 🌍 worlds_and_profiles | swap the fake chain for a real one under an *unchanged* lifecycle — the composition root, live | Anvil |
| 12 | 🎆 grand_finale | perform the whole play at maximum headless realism, revocation showpiece included — then sit the final exam | Anvil |
| 13 | 📏 the_evaluation | say what an existence proof does *not* prove, give the two definitions of "enforced", name the five simulation boundaries, and walk the harness's seven experiments | — |
| 14 | 🧾 results_and_conclusions | read the measured results, weigh the threats to validity, and say which feasibility conclusions the numbers do — and don't — support | — |

Every notebook is **self-contained**: its own imports, its own throwaway chain if it
needs one. Each was verified to run green, top to bottom, with no infra beyond the
"Extra infra" column — the exact check being:

```bash
uv run --group demo jupyter nbconvert --to notebook --execute --stdout \
    e2e/notebooks/course/00_orientation.ipynb > /dev/null
```

## 8. Your setup check — and what "optional infra" means

Everything in this course runs with nothing but the synced workspace. Four *optional*
pieces unlock more realism, and any course cell that wants a missing piece will
**detect the absence, print the enabling command, and skip — never crash**. That is
what "guarded" means when you meet it later. The pieces:

- **Anvil** — a disposable local blockchain, installed with
  [Foundry](https://getfoundry.sh) — plus the **forge artifacts** (compiled contracts,
  from `forge build --root contracts`) → unlock notebooks 06, 07, 11, 12.
- **The containerlab lab** — a real router OS in containers → unlocks the
  `netctl_explore` / `console_explore` deep dives; course notebook 09 only *recipes* it.
- **An LLM endpoint** — `.env` per `llmserve/README.md` → optional live judgment; the
  deterministic stand-ins are the default everywhere.

First the *required* part — the six Python trust domains must import. (This cell pulls
the upper layers into `sys.modules`, which is why section 5's purity proof had to run
*before* it. Order was load-bearing.)

In [ ]:
import importlib

missing = []
for mod in ["a2a_interfaces", "chainmcp", "netctl", "controller", "agents", "e2e"]:
    try:
        importlib.import_module(mod)
        print(f"✓ import {mod}")
    except ImportError as err:
        missing.append(mod)
        print(f"✗ import {mod}  ({err})")

assert sys.version_info >= (3, 12), "this repo needs Python 3.12+"
assert not missing, "run `uv sync --all-packages` at the repo root, then Restart + Run All"
print(f"\n✓ Python {sys.version.split()[0]} + all six Python trust domains import — required setup complete")

Now the optional pieces — your personal report card. A ✗ here is **normal**, not a
problem: every course notebook still runs; the guarded cells will simply tell you what
they skipped and how to enable it. (`lab_ipv4` shells out to docker, so we guard even
the *check* — infra probing can itself need infra.)

In [ ]:
import os
from chainmcp.testing import anvil_available, artifacts_available   # the repo's own guards

try:
    from netctl.testing import lab_ipv4
    lab_ip = lab_ipv4()                       # None unless the containerlab lab is up
except Exception:                             # e.g. docker itself not installed
    lab_ip = None
llm_hint = bool(os.environ.get("LLM_BASE_URL")) or (ROOT / ".env").exists()

checks = [
    ("anvil on PATH",   anvil_available(),     "notebooks 06, 07, 11, 12", "install Foundry → https://getfoundry.sh"),
    ("forge artifacts", artifacts_available(), "same four notebooks",      "forge build --root contracts"),
    ("containerlab lab", lab_ip is not None,   "netctl/console explore twins", "containerlab deploy -t netlab/topology.clab.yml"),
    ("LLM endpoint",    llm_hint,              "live judgment (optional everywhere)", "see llmserve/README.md"),
]
for name, ok, unlocks, fix in checks:
    mark = "✓" if ok else "✗"
    tail = "" if ok else f"   (enable: {fix})"
    print(f"{mark} {name:<16} → unlocks {unlocks}{tail}")

print("\nreport, not exam: ✗ lines are fine — guarded cells skip politely and name the fix")

**✏️ Your turn 8.1 — Predict your own score**

You just read your report card. Write into `my_prediction` how many of the four optional checks showed ✓ on YOUR machine (0–4), then compute the real count from the `checks` list. Success: the self-check passes — and any score is a fine score.

In [ ]:
my_prediction = -1     # TODO: your guess, 0–4

score = 0              # TODO: count the ok flags (hint: sum(1 for _, ok, _, _ in checks if ok))
print(f"optional infra score: {score}/4 — every course notebook runs at any score")

# TODO: un-comment once both TODOs are done:
# assert score == my_prediction, "your memory of your own report card disagrees with it"

<details><summary>✅ Solution 8.1 — peek only after trying</summary>

```python
my_prediction = 2      # whatever YOUR report card actually showed
score = sum(1 for _, ok, _, _ in checks if ok)
```

The same `checks` list that printed the report is plain data you can compute over — guards are ordinary Python, not notebook magic.

</details>

## What you learned (and where it goes)

| You did | The concept | Where it goes next |
|---|---|---|
| ran, broke, and read the output of cells | notebook, cell, kernel, traceback | every notebook from here on |
| recomputed Ada's 45 GB → 50 Mbps by hand | the canonical example's numbers are honest | 04 dissects every value |
| imported `a2a_interfaces` and found its file | module / package / import, src-layout, editable installs | 01–03 read this package line by line |
| listed workspace members; printed the import diagram; proved bedrock loads no machinery | trust domains + downward-only imports | 03 (the ports), 08 (the purity proof) |
| read the 10 rules out of `CLAUDE.md`; verified only `chainmcp` imports web3 | rules as *enforced* boundaries, not vibes | each rule is demonstrated in its own chapter |
| ran the guarded setup report | optional infra; what "guarded cell" means | 06/07/11/12 (Anvil) and the lab-bound explore twins |


## Experiments to try

Predict first, then run — the prediction is the exercise:

- **Restart the kernel**, then run *only* section 2's arithmetic cell. Predict the exact
  error type before you look (section 1 named it). Then Run All to heal everything.
- Change `dataset_gb` to 90 in section 1 and re-run sections 1–4. Predict which assert
  stops you first, and say what disagreement it just caught for you.
- Deliberate breakage: in section 5's `sys.modules` cell, first run section 8's import
  cell, then re-run the section 5 check. Predict which of the four names now trips the
  assert, and explain why cell *order* was part of the proof.
- In section 6's scanner, swap `"web3"` for `"pygnmi"` (the router-talking library).
  Predict which single package the scan will find before you run it.


## Check yourself

You own this notebook when you can answer these out loud:

1. Ada pays 10 TOK and receives ticket #7 in the *same transaction*. Why does "same
   transaction" matter more than "fast"?
2. In one sentence each: what may an LLM decide in this system — and what may it never
   decide?
3. Which file does `import a2a_interfaces` execute, and why doesn't it matter that the
   repo folder is named `interfaces/`?
4. Name three packages and, for each, the one dangerous thing it alone is trusted with.
5. Your laptop has no Anvil. Which course notebooks change behavior, and what exactly
   will their guarded cells do instead of crashing?

**Where this goes next** → [01_python_toolbox.ipynb](01_python_toolbox.ipynb): every
language construct the repo's code uses — type hints, decorators, dataclasses, Enums,
exceptions, context managers, a first thread — each from zero, each landing in a real
repo file.

**Deeper dives**

- [docs/00-the-story.md](../../../docs/00-the-story.md) — the full story: every concept, introduced by the problem it solves
- [docs/02-architecture.md](../../../docs/02-architecture.md) — the map, formally: the import graph and why it's shaped that way
- [docs/LEARNING-PATH.md](../../../docs/LEARNING-PATH.md) — the checkable reading route this course is the executable twin of
- [e2e/notebooks/skeleton_explore.ipynb](../skeleton_explore.ipynb) — the whole lifecycle on cardboard, compressed; course notebook 05 is its from-zero twin
